In [ ]:
# Optimisation du portefeuille 

# On possède 5 actions (ex: Apple, Microsoft, Google...)
# --> utiliser NumPy pour calculer la matrice de covariance de leurs rendements. 
# --> utiliser scipy.optimize.minimize pour trouver exactement le poids de chaque action 

# Objectif : maximiser le ratio de Sharpe (le meilleur rendement pour le risque le plus faible).

# Sharpe = (E(Rp) - Rf)/sigma
# --> E(Rp) rendement du portefeuille attendu
# --> Rf = taux sans risque (qu'on fixe à 0 pour simplifier)
# --> sigma = volatilité du portefeuille

In [11]:
import numpy as np
import pandas as pd
import scipy.optimize as sco

# 1. Génération des données fictives (yfinance)
# On simule les rendements journaliers de 4 actifs sur 5 ans (1260 jours de trading)

np.random.seed(42)

actions = ['AAPL', 'MSFT', 'GOOG', 'AMZN']
num_actions = len(actions)
returns = pd.DataFrame(np.random.normal(0.0005, 0.015, (1260, num_actions)), columns=actions)
# On simule 5 ans de trading et 252 jours de trading par an --> 1260

# 2. Calculs statistiques vectorisés
# Rendements annuels moyens (252 jours de trading par an)
mean_returns = returns.mean() * 252 
# * 252 pour être annuel

# Matrice de covariance annuelle
cov_matrix = returns.cov() * 252

# 3. Fonction objectif pour l'optimiseur
def portfolio_statistics(weights, mean_returns, cov_matrix):
    """Calcule le rendement, la volatilité et le ratio de Sharpe d'un portefeuille."""
    weights = np.array(weights)
    # Rendement du portefeuille : produit scalaire des poids et des rendements
    port_return = np.sum(mean_returns * weights)
    # Volatilité du portefeuille : sqrt(W^T * Cov * W)
    port_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    # Ratio de Sharpe (taux sans risque = 0)
    sharpe_ratio = port_return / port_volatility
    return port_return, port_volatility, sharpe_ratio

def min_func_sharpe(weights, mean_returns, cov_matrix):
    """Fonction à minimiser. Comme on veut MAXIMISER Sharpe, on MINIMISE (-Sharpe)."""
    # L'optimiseur scipy cherche toujours à minimiser
    return -portfolio_statistics(weights, mean_returns, cov_matrix)[2]

# 4. Contraintes et limites
# Contrainte : La somme des poids doit être égale à 1 (100% du capital alloué)
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

# Limites : Chaque poids doit être compris entre 0 et 1 (Pas de vente à découvert / short-selling)
bounds = tuple((0, 1) for asset in range(num_actions))

# Répartition initiale (égale partout) : 25% par actif
initial_guess = num_actions * [1. / num_actions]

# 5. OPTIMISATION
opts = sco.minimize(
    fun=min_func_sharpe, 
    x0=initial_guess, 
    args=(mean_returns, cov_matrix),
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
)

# 6. AFFICHAGE DES RÉSULTATS
optimal_weights = opts['x']
opt_return, opt_vol, opt_sharpe = portfolio_statistics(optimal_weights, mean_returns, cov_matrix)

print("=== ALLOCATION OPTIMALE DES POIDS ===")
for actions, weight in zip(actions, optimal_weights):
    print(f"{actions} : {weight*100:.2f}%")

print("\n=== PERFORMANCES ATTENDUES DU PORTEFEUILLE ===")
print(f"Rendement annuel attendu : {opt_return*100:.2f}%")
print(f"Volatilité annuelle attendue : {opt_vol*100:.2f}%")
print(f"Ratio de Sharpe Max : {opt_sharpe:.2f}")

=== ALLOCATION OPTIMALE DES POIDS ===
AAPL : 43.31%
MSFT : 24.48%
GOOG : 1.44%
AMZN : 30.77%

=== PERFORMANCES ATTENDUES DU PORTEFEUILLE ===
Rendement annuel attendu : 20.48%
Volatilité annuelle attendue : 13.55%
Ratio de Sharpe Max : 1.51
